## Task 1: Implementing TF-IDF Based Recommendation

In [5]:
pip install -r requirements.txt

  Using cached pandas-3.0.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 13.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 16.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 16.8 MB/s  0:00:00 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 16.2 MB/s  0:00:01m0:00:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv('data/movies.csv')

movies['genres_processed'] = movies['genres'].str.replace('|', ' ')

In [8]:
movies['genres_processed']

0       Adventure Animation Children Comedy Fantasy
1                        Adventure Children Fantasy
2                                    Comedy Romance
3                              Comedy Drama Romance
4                                            Comedy
                           ...                     
9737                Action Animation Comedy Fantasy
9738                       Animation Comedy Fantasy
9739                                          Drama
9740                               Action Animation
9741                                         Comedy
Name: genres_processed, Length: 9742, dtype: str

In [9]:
# Initialize
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_processed'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

TF-IDF Matrix Shape: (9742, 23)


In [10]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix) #applying cosine similarty between each movie with each movie

In [17]:
# Create a reverse mapping of movie titles to their indices
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def get_content_recommendations(title, cos_sim=cosine_sim, top_n=1):
    # 1. Get the index of the movie that matches the title
    if title not in indices:
        return "Movie not found in the dataset."
    movie_id = indices[title]

    cos_sim = list(enumerate(cos_sim[movie_id]))

    cos_sim = sorted(cos_sim, key=lambda x: x[1], reverse=True)

    cos_sim = cos_sim[1:top_n+1]

    movie_indices = [i[0] for i in cos_sim]
    scores = [i[1] for i in cos_sim]

    recommendations = pd.DataFrame({
        'Movie': movies['title'].iloc[movie_indices].values,
        'Similarity Score': scores
    })
    
    return recommendations

In [20]:

print(get_content_recommendations('Bridge of Spies (2015)', top_n=2))

               Movie  Similarity Score
0  Juror, The (1996)               1.0
1   City Hall (1996)               1.0


## Task 2: User-Profile-Based Content Recommender 

In [21]:
import numpy as np

ratings = pd.read_csv('data/ratings.csv')

target_user_id = 1
user_ratings = ratings[ratings['userId'] == target_user_id]

In [22]:
def build_user_profile(user_id, ratings_df, movies_df, tfidf_matrix):
    
    user_data = ratings_df[ratings_df['userId'] == user_id]
    
    profile_vector = np.zeros(tfidf_matrix.shape[1])
    total_rating_weight = 0
    
    for index, row in user_data.iterrows():
        movie_id = row['movieId']
        rating = row['rating']
    
        movie_idx_list = movies_df.index[movies_df['movieId'] == movie_id].tolist()
        
        if not movie_idx_list:
            continue 
            
        movie_idx = movie_idx_list[0]
        
        movie_vector = tfidf_matrix[movie_idx].toarray()[0]
        
        profile_vector += (movie_vector * rating)
        total_rating_weight += rating
        
    if total_rating_weight > 0:
        profile_vector = profile_vector / total_rating_weight
        
    return profile_vector

user_1_profile = build_user_profile(target_user_id, ratings, movies, tfidf_matrix)

In [23]:
def recommend_for_user(user_profile, tfidf_matrix, movies_df, top_n=5):

    user_profile_2d = user_profile.reshape(1, -1)
    
    similarity_scores = cosine_similarity(user_profile_2d, tfidf_matrix).flatten()
    
    top_indices = similarity_scores.argsort()[-top_n:][::-1]
    
    recommendations = movies_df.iloc[top_indices][['title', 'genres']]
    recommendations['Similarity'] = similarity_scores[top_indices]
    
    return recommendations

print(f"Top 5 Recommendations for User {target_user_id}:")
print(recommend_for_user(user_1_profile, tfidf_matrix, movies))

Top 5 Recommendations for User 1:
                                       title  \
8597   Dragonheart 2: A New Beginning (2000)   
6570               Hunting Party, The (2007)   
4005                        Flashback (1990)   
4681          The Great Train Robbery (1978)   
4409  Charlie's Angels: Full Throttle (2003)   

                                              genres  Similarity  
8597  Action|Adventure|Comedy|Drama|Fantasy|Thriller    0.791026  
6570          Action|Adventure|Comedy|Drama|Thriller    0.773645  
4005             Action|Adventure|Comedy|Crime|Drama    0.773089  
4681             Action|Adventure|Comedy|Crime|Drama    0.773089  
4409          Action|Adventure|Comedy|Crime|Thriller    0.760503  


In [25]:
def evaluate_user_profile(user_id, ratings_df, tfidf_matrix, movies_df, k=10, rating_threshold=4.0):
    
    user_ratings = ratings_df[ratings_df['userId'] == user_id]
    relevant_movies = user_ratings[user_ratings['rating'] >= rating_threshold]['movieId'].tolist()
    
    if len(relevant_movies) == 0:
        return "User has no highly rated movies to evaluate against."
    
    #Build the user's profile
    user_profile = build_user_profile(user_id, ratings_df, movies_df, tfidf_matrix)
    
    #Get Top-K Recommendations
    user_profile_2d = user_profile.reshape(1, -1)
    similarity_scores = cosine_similarity(user_profile_2d, tfidf_matrix).flatten()
    
    
    top_indices = similarity_scores.argsort()[::-1]
    
    
    recommended_movie_ids = movies_df.iloc[top_indices]['movieId'].values
    
    top_k_recommendations = recommended_movie_ids[:k]
    
    hits = set(top_k_recommendations).intersection(set(relevant_movies))
    num_hits = len(hits)
    
    precision_at_k = num_hits / k
    recall_at_k = num_hits / len(relevant_movies)
    
    print(f"--- Evaluation for User {user_id} at K={k} ---")
    print(f"Total Relevant Movies : {len(relevant_movies)}")
    print(f"Number of Hits in Top {k}: {num_hits}")
    print(f"Precision@{k}: {precision_at_k:.4f}")
    print(f"Recall@{k}: {recall_at_k:.4f}")
    
    return precision_at_k, recall_at_k

evaluate_user_profile(target_user_id, ratings, tfidf_matrix, movies, k=10)

--- Evaluation for User 1 at K=10 ---
Total Relevant Movies : 200
Number of Hits in Top 10: 0
Precision@10: 0.0000
Recall@10: 0.0000


(0.0, 0.0)